# Bước 3 – Data Analysis with SQL
## Olist Brazilian E-Commerce · Reverse Logistics Analysis

**Mục tiêu:** Dùng SQL (SQLite in-memory) để trả lời từng phần của 3 Research Questions:
- **RQ1 – Consumer Behavior:** Hành vi khách hàng & brand loyalty
- **RQ2 – Financial Impact:** Chi phí logistics & tác động tài chính
- **RQ3 – Technology Integration:** Hiệu suất logistics & nhu cầu ứng dụng công nghệ

> **Input:** `olist_analytical_table.csv` (output của Bước 2 – 99,441 rows × 46 cols)  
> **SQL file:** `queries.sql` (7 queries đầy đủ comment)

## 0. Setup & Load dữ liệu vào SQLite

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Import thư viện thành công!')

In [ ]:
# Load CSV vào SQLite in-memory database
# Dùng in-memory để không cần tạo file .db trên disk
df = pd.read_csv('olist_analytical_table.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('olist_analytical_table', conn, if_exists='replace', index=False)

print(f'Loaded {len(df):,} rows × {df.shape[1]} cols vào SQLite')
print(f'Các cột: {list(df.columns)}')

# Helper function: chạy SQL → trả về DataFrame
def run_sql(query, connection=conn):
    return pd.read_sql_query(query, connection)

---
## 1. RQ1 – Consumer Behavior

> **Research Question:** How does consumer perception of "sustainable return" policies and refurbished/recycled products in e-commerce affect their purchasing decisions and brand loyalty?

**Proxy từ Olist data (không có dữ liệu return trực tiếp):**
| Khái niệm | Proxy dùng |
|-----------|------------|
| Brand loyalty | Khách hàng mua lặp lại (`customer_unique_id` > 1 lần) |
| Purchasing decision | `avg_review_score` – quyết định mua lại hay không |
| Return trigger | Đơn giao trễ + review ≤ 2 sao |

**Queries:** Q1 (loyalty) · Q2 (late delivery → satisfaction)

### Query 1 – Tỉ lệ khách hàng quay lại (Brand Loyalty Proxy)

Phân loại khách hàng theo số lần mua để đo lường **brand loyalty**.  
Khách hàng mua ≥ 2 lần = loyal — chỉ số quan trọng khi doanh nghiệp triển khai circular economy.

**Kỳ vọng:** Tỉ lệ loyal thấp → cơ hội lớn để tăng retention thông qua chính sách đổi trả bền vững.

In [ ]:
q1 = '''
SELECT
    customer_type,
    COUNT(*)                               AS customer_count,
    ROUND(AVG(total_orders), 2)            AS avg_orders_per_customer,
    ROUND(AVG(avg_review_score), 2)        AS avg_satisfaction_score
FROM (
    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id)           AS total_orders,
        AVG(avg_review_score)              AS avg_review_score,
        CASE
            WHEN COUNT(DISTINCT order_id) > 1 THEN 'Loyal (Repeat buyer)'
            ELSE 'One-time buyer'
        END                                AS customer_type
    FROM olist_analytical_table
    WHERE order_status = 'delivered'
    GROUP BY customer_unique_id
) sub
GROUP BY customer_type
'''

df_q1 = run_sql(q1)
display(df_q1)

# ── Visualization ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Chart 1: Số lượng khách hàng
colors = ['#42A5F5', '#EF5350']
axes[0].bar(df_q1['customer_type'], df_q1['customer_count'], color=colors)
axes[0].set_title('Số lượng khách hàng theo loại', fontweight='bold')
axes[0].set_ylabel('Số khách hàng')
for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
        f'{int(bar.get_height()):,}', ha='center', fontsize=10
    )

# Chart 2: Điểm hài lòng trung bình
axes[1].bar(df_q1['customer_type'], df_q1['avg_satisfaction_score'],
            color=['#66BB6A', '#FFA726'])
axes[1].set_title('Điểm hài lòng trung bình', fontweight='bold')
axes[1].set_ylabel('Avg Review Score')
axes[1].set_ylim(0, 5)
for bar in axes[1].patches:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
        f'{bar.get_height():.2f}', ha='center', fontsize=11
    )

plt.suptitle('RQ1 – Q1: Brand Loyalty Proxy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Query 2 – Tác động giao hàng trễ đến sự hài lòng khách hàng

Giao hàng trễ là **reverse logistics trigger** phổ biến nhất.  
Khách hàng không hài lòng (review ≤ 2) → khả năng trả hàng cao và không mua lại.

**Metric chính:** `low_score_pct` = % đơn bị đánh giá ≤ 2 sao theo trạng thái giao hàng.

In [ ]:
q2 = '''
SELECT
    CASE WHEN is_late_delivery = 1 THEN 'Giao tre' ELSE 'Dung han' END  AS delivery_status,
    COUNT(*)                                                             AS order_count,
    ROUND(AVG(avg_review_score), 2)                                      AS avg_review_score,
    ROUND(AVG(delay_days), 1)                                            AS avg_delay_days,
    SUM(CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END)              AS low_score_count,
    ROUND(
        100.0 * SUM(CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END)
        / COUNT(*), 2
    )                                                                    AS low_score_pct
FROM olist_analytical_table
WHERE is_late_delivery IS NOT NULL
GROUP BY is_late_delivery
'''

df_q2 = run_sql(q2)
display(df_q2)

# ── Visualization ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#66BB6A', '#EF5350']

# Chart 1: Avg review score
axes[0].bar(df_q2['delivery_status'], df_q2['avg_review_score'], color=colors)
axes[0].set_title('Avg Review Score: Đúng hạn vs Giao trễ', fontweight='bold')
axes[0].set_ylim(0, 5)
axes[0].set_ylabel('Review Score')
for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
        f'{bar.get_height():.2f}', ha='center', fontsize=11
    )

# Chart 2: Tỉ lệ review thấp
axes[1].bar(df_q2['delivery_status'], df_q2['low_score_pct'], color=colors)
axes[1].set_title('Tỉ lệ đánh giá thấp (≤ 2 sao)', fontweight='bold')
axes[1].set_ylabel('%')
for bar in axes[1].patches:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        f'{bar.get_height():.1f}%', ha='center', fontsize=11
    )

plt.suptitle('RQ1 – Q2: Giao trễ ảnh hưởng đến Consumer Satisfaction',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. RQ2 – Financial Impact

> **Research Question:** How does the transition from traditional reverse logistics to a closed-loop supply chain impact the cost structure and profit margins of e-commerce businesses?

**Proxy từ Olist data:**
| Khái niệm | Proxy dùng |
|-----------|------------|
| Cost structure | `total_freight / total_price` (freight ratio %) |
| Revenue loss | Doanh thu từ đơn canceled / unavailable |
| Profit margin trends | Doanh thu giao thành công theo tháng |

**Queries:** Q3 (freight ratio by category) · Q4 (cancellation cost) · Q5 (monthly trend)

### Query 3 – Chi phí logistics theo danh mục sản phẩm (Freight Ratio)

`freight_ratio_pct` = tỉ lệ phí vận chuyển / giá sản phẩm.  
Danh mục có **freight ratio cao** → chi phí logistics lớn → ưu tiên tối ưu hóa trong closed-loop supply chain.  

**Chỉ tính đơn delivered và danh mục có ≥ 50 đơn** để đảm bảo đủ mẫu.

In [ ]:
q3 = '''
SELECT
    product_category_name_english                                          AS category,
    COUNT(*)                                                               AS order_count,
    ROUND(AVG(total_price),   2)                                           AS avg_product_price,
    ROUND(AVG(total_freight), 2)                                           AS avg_freight_cost,
    ROUND(
        100.0 * AVG(total_freight) / NULLIF(AVG(total_price), 0), 2
    )                                                                      AS freight_ratio_pct
FROM olist_analytical_table
WHERE order_status     = 'delivered'
  AND product_category_name_english IS NOT NULL
GROUP BY product_category_name_english
HAVING COUNT(*) > 50
ORDER BY freight_ratio_pct DESC
LIMIT 15
'''

df_q3 = run_sql(q3)
display(df_q3)

# ── Visualization ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
df_plot = df_q3.sort_values('freight_ratio_pct')
bars = ax.barh(
    df_plot['category'], df_plot['freight_ratio_pct'],
    color=sns.color_palette('RdYlGn_r', len(df_plot))
)
for bar in bars:
    ax.text(
        bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
        f'{bar.get_width():.1f}%', va='center', fontsize=9
    )
ax.set_title('Top 15 danh mục: Freight Cost / Product Price (%)',
             fontweight='bold')
ax.set_xlabel('Freight Ratio (%)')
plt.tight_layout()
plt.show()

### Query 4 – Tác động tài chính của từng trạng thái đơn hàng

Đơn `canceled` và `unavailable` = **doanh thu bị mất** + chi phí logistics lãng phí.  
`total_value_impact` = tổng giá trị tài chính bị ảnh hưởng theo từng trạng thái.

In [ ]:
q4 = '''
SELECT
    order_status,
    COUNT(*)                              AS order_count,
    ROUND(AVG(total_order_value), 2)      AS avg_order_value,
    ROUND(SUM(total_order_value), 2)      AS total_value_impact,
    ROUND(AVG(total_freight),     2)      AS avg_freight_cost,
    ROUND(AVG(avg_review_score),  2)      AS avg_review_score
FROM olist_analytical_table
GROUP BY order_status
ORDER BY order_count DESC
'''

df_q4 = run_sql(q4)
display(df_q4)

# ── Tổng doanh thu bị ảnh hưởng từ đơn không delivered ────────
bad_statuses = ['canceled', 'unavailable', 'processing', 'created', 'approved']
bad = df_q4[df_q4['order_status'].isin(bad_statuses)]
print(f"\nTổng giá trị đơn không delivered: R$ {bad['total_value_impact'].sum():,.2f}")

### Query 5 – Xu hướng doanh thu và tỉ lệ hủy đơn theo tháng

Theo dõi **xu hướng thời gian** để:
- Xác định mùa cao điểm cần đầu tư reverse logistics
- Đánh giá tỉ lệ hủy có xu hướng tăng/giảm theo thời gian

In [ ]:
q5 = '''
SELECT
    purchase_year,
    purchase_month,
    COUNT(*)                                                                      AS total_orders,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END)                   AS canceled_orders,
    ROUND(
        100.0 * SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    )                                                                             AS cancel_rate_pct,
    ROUND(
        SUM(CASE WHEN order_status = 'delivered' THEN total_order_value ELSE 0 END), 2
    )                                                                             AS delivered_revenue
FROM olist_analytical_table
GROUP BY purchase_year, purchase_month
ORDER BY purchase_year, purchase_month
'''

df_q5 = run_sql(q5)
df_q5['period'] = pd.to_datetime(
    df_q5['purchase_year'].astype(str) + '-' +
    df_q5['purchase_month'].astype(str).str.zfill(2)
)

display(df_q5.head(10))

# ── Visualization ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Chart 1: Doanh thu hàng tháng
axes[0].bar(df_q5['period'], df_q5['delivered_revenue'] / 1e6,
            color='#42A5F5', width=20)
axes[0].set_title('Doanh thu từ đơn delivered theo tháng (triệu R$)',
                  fontweight='bold')
axes[0].set_ylabel('Triệu R$')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:.1f}M')
)

# Chart 2: Tỉ lệ hủy đơn
axes[1].plot(df_q5['period'], df_q5['cancel_rate_pct'],
             marker='o', color='#EF5350', linewidth=2)
axes[1].fill_between(df_q5['period'], df_q5['cancel_rate_pct'],
                     alpha=0.15, color='#EF5350')
axes[1].set_title('Tỉ lệ hủy đơn theo tháng (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle('RQ2 – Q5: Xu hướng Doanh thu & Tỉ lệ Hủy đơn theo Tháng',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. RQ3 – Technology Integration

> **Research Question:** What role do digital technologies (such as AI, IoT, or Blockchain) play in enhancing the transparency of product recovery and improving sorting efficiency within the reverse logistics chain?

**Proxy từ Olist data:**
| Khái niệm | Proxy dùng |
|-----------|------------|
| AI prediction accuracy | Khoảng cách `estimated_delivery_days` vs `actual_delivery_days` |
| IoT tracking need | Sellers có `late_rate_pct` cao = thiếu visibility thời gian thực |
| Blockchain transparency | Sellers có review thấp + giao trễ = cần audit trail minh bạch |

**Queries:** Q6 (logistics timing) · Q7 (seller risk nodes)

### Query 6 – Phân tích thời gian các giai đoạn logistics

So sánh `actual_delivery_days` vs `estimated_delivery_days` theo từng trạng thái đơn.  
Khoảng cách lớn = **hệ thống dự đoán kém** → cần AI/ML để tối ưu dự báo (RQ3 technology gap).  

`avg_delay_days` âm = giao **sớm hơn** ước tính (tốt), dương = giao **trễ hơn** (xấu).

In [ ]:
q6 = '''
SELECT
    order_status,
    COUNT(*)                                     AS order_count,
    ROUND(AVG(actual_delivery_days),    1)        AS avg_actual_days,
    ROUND(AVG(estimated_delivery_days), 1)        AS avg_estimated_days,
    ROUND(AVG(delay_days),              1)        AS avg_delay_days,
    ROUND(MIN(actual_delivery_days),    0)        AS min_days,
    ROUND(MAX(actual_delivery_days),    0)        AS max_days
FROM olist_analytical_table
WHERE actual_delivery_days IS NOT NULL
GROUP BY order_status
ORDER BY order_count DESC
'''

df_q6 = run_sql(q6)
display(df_q6)

# ── Gap: Ước tính vs Thực tế (chỉ delivered) ──────────────────
delivered_row = df_q6[df_q6['order_status'] == 'delivered'].iloc[0]
print(f"\n[Delivered orders]")
print(f"  Ước tính trung bình : {delivered_row['avg_estimated_days']:.1f} ngày")
print(f"  Thực tế trung bình  : {delivered_row['avg_actual_days']:.1f} ngày")
print(f"  Giao sớm hơn TB     : {abs(delivered_row['avg_delay_days']):.1f} ngày")

### Query 7 – Nhận diện Seller có hiệu suất logistics kém (High-Risk Nodes)

Seller với `late_rate_pct` cao và `avg_review_score` thấp = **nút rủi ro** trong supply chain.  
Đây là đối tượng cần ứng dụng:
- **IoT** → theo dõi hàng hóa thời gian thực, phát hiện sớm rủi ro giao trễ
- **Blockchain** → audit trail minh bạch lịch sử vận chuyển
- **AI** → dự đoán và cảnh báo trước nguy cơ chậm trễ

_Chỉ lấy seller có ≥ 20 đơn để đảm bảo thống kê đáng tin cậy._

In [ ]:
q7 = '''
SELECT
    seller_id,
    seller_state,
    COUNT(*)                                                              AS total_orders,
    SUM(CASE WHEN is_late_delivery = 1 THEN 1 ELSE 0 END)                AS late_orders,
    ROUND(
        100.0 * SUM(CASE WHEN is_late_delivery = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 2
    )                                                                     AS late_rate_pct,
    ROUND(AVG(avg_review_score), 2)                                       AS avg_review_score,
    ROUND(AVG(delay_days),       1)                                       AS avg_delay_days
FROM olist_analytical_table
WHERE seller_id        IS NOT NULL
  AND is_late_delivery IS NOT NULL
GROUP BY seller_id, seller_state
HAVING COUNT(*) >= 20
ORDER BY late_rate_pct DESC
LIMIT 20
'''

df_q7 = run_sql(q7)
display(df_q7)

# ── Visualization: Top 10 sellers rủi ro cao nhất ──────────────
df_top10 = df_q7.head(10).copy()
df_top10['seller_label'] = df_top10['seller_id'].str[:8] + '..'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Late rate
axes[0].bar(
    df_top10['seller_label'], df_top10['late_rate_pct'],
    color=sns.color_palette('Reds_r', len(df_top10))
)
axes[0].set_title('Top 10 Sellers: Tỉ lệ giao trễ (%)', fontweight='bold')
axes[0].set_ylabel('Late Delivery Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

# Chart 2: Review score tương ứng
axes[1].bar(
    df_top10['seller_label'], df_top10['avg_review_score'],
    color=sns.color_palette('Blues_r', len(df_top10))
)
axes[1].set_title('Avg Review Score của Top 10 Sellers trễ nhất',
                  fontweight='bold')
axes[1].set_ylabel('Review Score')
axes[1].set_ylim(0, 5)
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('RQ3 – Q7: High-Risk Logistics Nodes cần áp dụng IoT/Blockchain/AI',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Tóm tắt kết quả SQL Analysis

| RQ | Query | Insight chính |
|-----|-------|---------------|
| RQ1 | Q1 – Brand Loyalty | Đa số khách hàng chỉ mua 1 lần → cơ hội lớn để circular economy tăng retention |
| RQ1 | Q2 – Late → Review | Giao trễ → review giảm đáng kể, tỉ lệ review ≤2 sao tăng mạnh |
| RQ2 | Q3 – Freight Ratio | Một số danh mục có freight ratio > 30% → chi phí logistics rất cao |
| RQ2 | Q4 – Cancellation Cost | 625 đơn canceled + 609 unavailable → doanh thu bị ảnh hưởng đáng kể |
| RQ2 | Q5 – Monthly Trend | Doanh thu tăng mạnh 2017→2018, tỉ lệ hủy ổn định ở mức thấp |
| RQ3 | Q6 – Logistics Timing | Giao hàng thực tế nhanh hơn ước tính ~11 ngày → hệ thống ước tính chưa tối ưu |
| RQ3 | Q7 – Seller Risk | Một số sellers có late rate > 50% → cần IoT/Blockchain giám sát trực tiếp |

> **Next step:** Bước 4 – Python EDA sâu hơn với statistical analysis và machine learning